#  ```insert_subgraph_with_mappings```
This example demonstrates how to convert a ``Neg`` node to a ``Mul`` node with -1. It provides a practical introduction to using the ``insert_subgraph_with_mappings`` API, highlighting important considerations when performing such graph surgery. <br><b>This method requires PyTorch to be installed, as it uses PyTorch to generate the replacement ONNX subgraph.</b>

## <b>Setup</b>

In [1]:
%pip install torch onnx onnx-graphsurgeon numpy onnxruntime torch netron

Note: you may need to restart the kernel to use updated packages.


In [6]:
# Step 1: Setup and Imports
import onnx
import onnx_graphsurgeon as gs
import logging

import tempfile
import os
import torch
import torch.nn as nn

# Optional: Set logging level for easier debugging
logging.basicConfig(level=logging.INFO)

## <b>Problem</b>
This code finds every ``Neg`` node in an ONNX graph and replaces it with a ``Mul`` node that multiplies by -1. For each replacement, it generates a small ONNX subgraph using PyTorch, ensures all names are unique, and uses the ``insert_subgraph_with_mappings`` API to perform the swap.

## <b>Code Flow</b>
Below is a step-by-step example showing how you can use this API to customize your ONNX graphs for your own needs. With just a basic understanding of ONNX GraphSurgeon/ Pytorch, you can easily perform targeted graph surgery and optimize your models, no need to retrain or rebuild from scratch!

### <b>Step 0 : Create/Load the model</b>
Load or create your model which needs to be modified

In [11]:
model = onnx.load("./example_models/complex_negtomul.onnx") # <- Replace with your ONNX model path
graph = gs.import_onnx(model)

import netron
# Launch Netron to visualize the ONNX model in your browser
netron.start("./example_models/complex_negtomul.onnx")


INFO:netron.server:Serving './example_models/complex_negtomul.onnx' at http://localhost:22584


('localhost', 22584)

### <b>Step 1 : Load the API</b>
Load the API from ```common.py``` 

In [8]:
from osrt_model_tools.onnx_tools.tidl_onnx_model_optimizer.src.common import insert_subgraph_with_mappings

### <b>Step 2 : Define A Custom Transformation Function</b>

In [12]:
def neg_to_mul(graph: gs.Graph):
    """
    Tutorial Example:
    Replace all Neg nodes in the graph with an equivalent Mul(-1) subgraph.
    This demonstrates how to use PyTorch to generate a replacement ONNX subgraph,
    and how to use the insert_subgraph_with_mappings API for graph surgery.
    """
    nodes = graph.nodes
    neg_idx = 0  # To ensure unique names for each replacement

    for node in nodes:
        if node.op == "Neg": # or have pattern matching here
            # --- 1. Gather info about the Neg node to be replaced ---
            input_name = node.inputs[0].name
            output_name = node.outputs[0].name
            input_shape = node.inputs[0].shape
            output_shape = node.outputs[0].shape
            suffix = f"_neg2mul_{neg_idx}"  # Unique suffix for this replacement

            # --- 2. Dynamically create a Mul(-1) ONNX subgraph using PyTorch ---
            class NegToMulModule(nn.Module):
                def __init__(self):
                    super().__init__() # Initialize the parent class
                def forward(self, x):
                    return x * -1

            dummy_input = torch.randn(*input_shape)  # Shape of the input/output tensors must match the Neg node's input
            try:
                model = NegToMulModule()
                y = model(dummy_input)
            except Exception as e:
                print(f"Error in PyTorch model creation for Neg node {node.name}. Please check the input shape: {input_shape}. Error: {e}")
                continue
                
            # Export the PyTorch module to a temporary ONNX file
            with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as tmpfile:
                torch.onnx.export(
                    NegToMulModule(),
                    dummy_input,
                    tmpfile.name,
                    input_names=["input"],
                    output_names=["output"],
                    dynamic_axes=None,
                    opset_version=graph.opset
                )
                mul_model = onnx.load(tmpfile.name)
                mul_model = onnx.shape_inference.infer_shapes(mul_model)  # Perform shape inference to ensure correct shapes
            os.remove(tmpfile.name)  # Clean up the temp file

            # --- 3. Import the ONNX subgraph into GraphSurgeon ---
            mul_gs = gs.import_onnx(mul_model)

            # Map the subgraph's input/output to the original graph's tensors                !IMP!
            input_mapping = {mul_gs.inputs[0].name: input_name}
            output_mapping = {mul_gs.outputs[0].name: output_name}
            
            print(f"Input mapping: {input_mapping}")
            print(f"Output mapping: {output_mapping}")


            # --- 5. Use the API to insert the new subgraph in place of the Neg node ---
            success = insert_subgraph_with_mappings(
                graph,
                input_mapping,
                output_mapping,
                mul_gs,
                suffix      # Optional: suffix for the new subgraph, handle renaming of nodes and tensors if not provided
            )
            if success:
                print(f"Neg→Mul(-1) replacement succeeded for node {node.name}.")
            else:
                print(f"Neg→Mul(-1) replacement failed for node {node.name}.")
            neg_idx += 1  # Increment for the next Neg node   

### <b>Step 3 : Call the function</b>
Pass the original graph to the function and save the transformed graph, if required

In [13]:
# Call the function            
neg_to_mul(graph)
onnx.save(gs.export_onnx(graph), "./example_models/optim_negtomul.onnx")
netron.start("./example_models/optim_negtomul.onnx")

INFO:netron.server:Serving './example_models/optim_negtomul.onnx' at http://localhost:23486


Input mapping: {'input': 'input'}
Output mapping: {'output': 'neg1_out'}
Neg→Mul(-1) replacement succeeded for node .
Input mapping: {'input': 'bn_out'}
Output mapping: {'output': 'neg2_out'}
Neg→Mul(-1) replacement succeeded for node .
Input mapping: {'input': 'relu1_out'}
Output mapping: {'output': 'neg3_out'}
Neg→Mul(-1) replacement succeeded for node .
Input mapping: {'input': 'add2_out'}
Output mapping: {'output': 'neg4_out'}
Neg→Mul(-1) replacement succeeded for node .
Input mapping: {'input': 'relu2_out'}
Output mapping: {'output': 'neg5_out'}
Neg→Mul(-1) replacement succeeded for node .
Input mapping: {'input': 'final_out'}
Output mapping: {'output': 'neg6_out'}
Neg→Mul(-1) replacement succeeded for node .


('localhost', 23486)